In [48]:
import numpy as np 
import pandas as pd
import xarray as xr

In [2]:
#load file for all dates
data_filepath = 'Verbose.csv'
df_data = pd.read_csv(data_filepath)
df_data

/var/folders/6m/87kslmb13vl09hjpjjpddb2h0000gn/T/ipykernel_37512/211380307.py:3: DtypeWarning: Columns (14,22,30) have mixed types. Specify dtype option on import or set low_memory=False.
  df_data = pd.read_csv(data_filepath)


,State Code,County Code,Site ID,Parameter,POC,Sample Duration,Unit,Method,Date,Start Time,...,SiteName,ParamMethod,AnalysisDescription,ParameterName,ChemicalFormula,dt,Sample End Time,Qualifier 1 Description,Qualifier 2 Description,Qualifier 3 Description
0,49.0,11.0,4.0,43102.0,1.0,1.0,78.0,228.0,20241001.0,00:00,...,NaN,43102.0228,NaN,NaN,NaN,2024-10-01 00:00:00,2024-10-01 01:00:00,NaN,NaN,NaN
1,49.0,11.0,4.0,43102.0,1.0,1.0,78.0,228.0,20241001.0,01:00,...,NaN,43102.0228,NaN,NaN,NaN,2024-10-01 01:00:00,2024-10-01 02:00:00,NaN,NaN,NaN
2,49.0,11.0,4.0,43102.0,1.0,1.0,78.0,228.0,20241001.0,02:00,...,NaN,43102.0228,NaN,NaN,NaN,2024-10-01 02:00:00,2024-10-01 03:00:00,NaN,NaN,NaN
3,49.0,11.0,4.0,43102.0,1.0,1.0,78.0,228.0,20241001.0,03:00,...,NaN,43102.0228,NaN,NaN,NaN,2024-10-01 03:00:00,2024-10-01 04:00:00,NaN,NaN,NaN
4,49.0,11.0,4.0,43102.0,1.0,1.0,78.0,228.0,20241001.0,04:00,...,NaN,43102.0228,NaN,NaN,NaN,2024-10-01 04:00:00,2024-10-01 05:00:00,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1614013,49.0,45.0,4.0,45225.0,1.0,1.0,78.0,228.0,20250228.0,20:00,...,NaN,45225.0228,NaN,NaN,NaN,2025-02-28 20:00:00,2025-02-28 21:00:00,Analyte Identified; Reported Value May Be Bias...,Value less than MDL.,NaN
1614014,49.0,45.0,4.0,45225.0,1.0,1.0,78.0,228.0,20250228.0,21:00,...,NaN,45225.0228,NaN,NaN,NaN,2025-02-28 21:00:00,2025-02-28 22:00:00,Analyte Identified; Reported Value May Be Bias...,Value less than MDL.,NaN
1614015,49.0,45.0,4.0,45225.0,1.0,1.0,78.0,228.0,20250228.0,22:00,...,NaN,45225.0228,NaN,NaN,NaN,2025-02-28 22:00:00,2025-02-28 23:00:00,Analyte Identified; Reported Value May Be Bias...,Value less than MDL.,NaN
1614016,49.0,45.0,4.0,45225.0,1.0,1.0,78.0,228.0,20250228.0,23:00,...,NaN,45225.0228,NaN,NaN,NaN,2025-02-28 23:00:00,2025-03-01 00:00:00,Analyte Identified; Reported Value May Be Bias...,Value less than MDL.,NaN


In [3]:
df_hawthorne_usos_only = df_data.loc[(df_data['StationSym'] == 'HW')] #Get only the data at Hawthorne
#set index to datetimeindex
df_hawthorne_usos_only.set_index(['dt'], inplace = True) 
df_hawthorne_usos_only.index = pd.to_datetime(df_hawthorne_usos_only.index)

#Get only the data during the USOS campaign
df_hawthorne_usos_only = df_hawthorne_usos_only.sort_index().loc['2024-07-14 00:00:00':'2024-08-18 23:00:00']

In [13]:
df_hawthorne_usos_only['Parameter']
df_hawthorne_usos_only.Parameter = df_hawthorne_usos_only.Parameter.astype(float)
pd.options.display.float_format = '{:.0f}'.format
df_hawthorne_usos_only.Parameter

dt
2024-07-14 00:00:00   43214
2024-07-14 00:00:00   45201
2024-07-14 00:00:00   45210
2024-07-14 00:00:00   43248
2024-07-14 00:00:00   43226
                       ... 
2024-08-18 23:00:00   43230
2024-08-18 23:00:00   43248
2024-08-18 23:00:00   43242
2024-08-18 23:00:00   45218
2024-08-18 23:00:00   45220
Name: Parameter, Length: 50148, dtype: float64

In [31]:
df_hawthorne_usos_only.Parameter = df_hawthorne_usos_only.Parameter.astype(int)
sorted_parameters = sorted(df_hawthorne_usos_only.Parameter.unique())
print(sorted_parameters)

[43102, 43141, 43202, 43203, 43204, 43205, 43206, 43212, 43214, 43216, 43217, 43218, 43220, 43221, 43224, 43226, 43227, 43230, 43231, 43232, 43233, 43235, 43238, 43242, 43243, 43244, 43245, 43247, 43248, 43249, 43250, 43252, 43253, 43261, 43262, 43263, 43280, 43284, 43285, 43291, 43502, 43954, 43960, 45109, 45201, 45202, 45203, 45204, 45207, 45208, 45209, 45210, 45211, 45212, 45213, 45218, 45219, 45220, 45225]


In [37]:
mapping_filepath = 'variables_epa_species.csv'
df_epa_var_mapping = pd.read_csv(mapping_filepath)

#Maps the provided variable names from CSV (Excel Spreadsheet) to Parameter so that we can identify different species
mapping_dict = {}
for param, name in zip(sorted_parameters, df_epa_var_mapping['Var_EPA'].values):
    mapping_dict[param] = name
df_hawthorne_usos_only['Species_name'] = df_hawthorne_usos_only['Parameter'].map(mapping_dict)
df_hawthorne_usos_only['Species_name']

dt
2024-07-14 00:00:00           Isobutane
2024-07-14 00:00:00             Benzene
2024-07-14 00:00:00    Isopropylbenzene
2024-07-14 00:00:00         Cyclohexane
2024-07-14 00:00:00       trans2Pentene
                             ...       
2024-08-18 23:00:00      3Methylpentane
2024-08-18 23:00:00         Cyclohexane
2024-08-18 23:00:00        Cyclopentane
2024-08-18 23:00:00    m_Diethylbenzene
2024-08-18 23:00:00             Styrene
Name: Species_name, Length: 50148, dtype: object

In [46]:
hawthorne_isoprene = df_hawthorne_usos_only.loc[df_hawthorne_usos_only['Species_name'] == 'Isoprene']
hawthorne_isoprene['Sample Value']

dt
2024-07-14 00:00:00   NaN
2024-07-14 01:00:00   NaN
2024-07-14 02:00:00   NaN
2024-07-14 03:00:00   NaN
2024-07-14 04:00:00   NaN
                       ..
2024-08-18 19:00:00   NaN
2024-08-18 20:00:00   NaN
2024-08-18 21:00:00   NaN
2024-08-18 22:00:00   NaN
2024-08-18 23:00:00   NaN
Name: Sample Value, Length: 864, dtype: float64

In [51]:
#load file for all dates
all_days_filepath = '../../USOS_shared/CampaignData_and_Merges/R0/CSL_MobileLab_Parked/merged/rev_30min/all_CSL_MobileLab_Parked_rev30minv4.nc'
all_days_filepath_load = xr.open_dataset(all_days_filepath)
df_alldays = all_days_filepath_load.to_dataframe()
df_alldays.reset_index(inplace=True)
df_alldays.set_index('time_local', inplace=True, drop=False)

In [52]:
df_july_dates_only = df_alldays.sort_index().loc["2024-07-14 18:00:00":"2024-07-31 23:30:00"]
df_august_dates_only = df_alldays.sort_index().loc["2024-08-01":"2024-08-18 17:30:00"]

#add july and august ozone species to df
df_july_species = pd.DataFrame()
df_august_species = pd.DataFrame()

df_july_species['O3'] = df_july_dates_only['O3_ppbv'].to_frame()
df_august_species['O3']= df_august_dates_only['O3_ppbv'].to_frame()

usos_varlist = ['Isoprene_PTR','Benzene_PTR','Toluene_PTR','Styrene_PTR','HCHO_CRDS']
usos_species_list = ['Isoprene', 'Benzene', 'Toluene', 'Styrene', 'Formaldehyde']

#add july and august halogen species to df
for species in range(0,5):
    df_july_species[usos_species_list[species]] = df_july_dates_only[usos_varlist[species]].to_frame()
    df_august_species[usos_species_list[species]] = df_august_dates_only[usos_varlist[species]].to_frame()

In [54]:
df_july_species.resample('H')

In [ ]:
new_index = pd.date_range(start=df_july_species.index.min(), end=df_july_species.index.max(), freq='1H')

# Reindex the DataFrame to one-hour intervals
df_hourly = df_july_species.reindex(new_index)

print(df_hourly)